In [1]:
%run "../0 - Base/inicializa.ipynb"

In [2]:
DF_Perdas = pd.read_csv("..\\Raw\\RAW_CADERNO_PERDAS.csv")

In [3]:
DF_Perdas.head(10)

,DATA_OCORRENCIA,SETOR_ORIGEM,DESCRICAO_MOTIVO_TEXTO_LIVRE,COD_PROD_OU_NOME,QTD_AVARIADA,VALOR_ESTIMADO_PREJUIZO,REGISTRADO_POR
0,01/01/2026,DEPOSITO,Falta de mercadoria no estoque - suspeita furto,601,1.10,"R$ 14,18",Diego
1,03/01/2026,HORTI,podre descartado no lixo,601,6.92,"R$ 100,71",Betao
2,04/01/2026,Açougue,Lata amassada no descarregamento,DIFERENCA CAIXA,7.95,"R$ 98,53",Betao
3,06/01/2026,CARNES,Diferenca de peso na balanca,501,1.07,"R$ 19,3",DESCONHECIDO
4,07/01/2026,HORTI,Sobra de pao dormido,PAO,4.59,"R$ 158,44",Diego
5,10/01/2026,Açougue,quebra de caixa operadora Juliana,101,1.13,"R$ 136,15",Juliana
6,12/01/2026,Padaria,Diferenca de peso na balanca,501,11.24,"R$ 64,3",Carlos
7,14/01/2026,hortifruti,podre descartado no lixo,201,12.60,"R$ 155,74",Diego
8,17/01/2026,Açougue,produto vencido na prateleira sem etiqueta,601,11.76,"R$ 67,76",Carlos
9,18/01/2026,frente de caixa,Falta de mercadoria no estoque - suspeita furto,101,8.83,"R$ 138,33",Carlos


#### Valida NULL

In [4]:
DF_Perdas['DATA_OCORRENCIA'].isna().value_counts()

DATA_OCORRENCIA
False    46
Name: count, dtype: int64

In [5]:
DF_Perdas['SETOR_ORIGEM'].isna().value_counts()

SETOR_ORIGEM
False    46
Name: count, dtype: int64

####   # Tem null

In [6]:
DF_Perdas['COD_PROD_OU_NOME'].isna().value_counts()  # TEM NULL

COD_PROD_OU_NOME
False    39
True      7
Name: count, dtype: int64

In [7]:
DF_Perdas['QTD_AVARIADA'].isna().value_counts()

QTD_AVARIADA
False    46
Name: count, dtype: int64

In [8]:
DF_Perdas['VALOR_ESTIMADO_PREJUIZO'].isna().value_counts()

VALOR_ESTIMADO_PREJUIZO
False    46
Name: count, dtype: int64

In [9]:
DF_Perdas['REGISTRADO_POR'].isna().value_counts()

REGISTRADO_POR
False    46
Name: count, dtype: int64

----

## ETL com loop

In [10]:
silverPerdas = DF_Perdas

In [11]:
silverPerdas = silverPerdas.rename(columns={
    'COD_PROD_OU_NOME':'COD_PROD',
    'REGISTRADO_POR':'RESPONSAVEL',
    'VALOR_ESTIMADO_PREJUIZO':'PREJUIZO_ESTIMADO',
    'DESCRICAO_MOTIVO_TEXTO_LIVRE':'DESCRICAO'
})

In [12]:
# silverPerdas['COD_PROD'] = (
#     silverPerdas['COD_PROD']
#     .fillna(1000001) # Trata valores nulos cod_prod
#     .replace(0.0, 1000001) # Limpa nulos e substitui 1000001
#     .replace('DIFERENCA CAIXA', 1000001)
#     .replace('PICANHA', 201)
#     .replace('PAO', 601)
# )

silverPerdas['COD_PROD'] = (
      silverPerdas['COD_PROD'].astype(str).str.strip().str.upper()
      .replace({'DIFERENCA CAIXA': '1000001', 'PICANHA': '201', 'PAO': '601'})
  )
silverPerdas['COD_PROD'] = (
      pd.to_numeric(silverPerdas['COD_PROD'], errors='coerce')
      .replace(0, 1000001).fillna(1000001).astype('Int64')
  )

In [13]:
silverPerdas['COD_PROD'].value_counts()

COD_PROD
1000001    14
601        11
101         6
201         5
501         4
102         3
301         3
Name: count, dtype: Int64

In [14]:
# Lista com o nome das colunas
colunas = ['SETOR_ORIGEM', 'RESPONSAVEL', 'DESCRICAO']

# Aplica .str.upper() em cada coluna individualmente
# for col in colunas:
#     silverPerdas[col] = silverPerdas[col].astype(str).str.upper()

for col in ['SETOR_ORIGEM', 'RESPONSAVEL', 'DESCRICAO']:
    silverPerdas[col] = silverPerdas[col].str.strip().str.upper()

# Exibe o resultado
silverPerdas[colunas].head(3)

,SETOR_ORIGEM,RESPONSAVEL,DESCRICAO
0,DEPOSITO,DIEGO,FALTA DE MERCADORIA NO ESTOQUE - SUSPEITA FURTO
1,HORTI,BETAO,PODRE DESCARTADO NO LIXO
2,AÇOUGUE,BETAO,LATA AMASSADA NO DESCARREGAMENTO


In [15]:
silverPerdas.head(5)

,DATA_OCORRENCIA,SETOR_ORIGEM,DESCRICAO,COD_PROD,QTD_AVARIADA,PREJUIZO_ESTIMADO,RESPONSAVEL
0,01/01/2026,DEPOSITO,FALTA DE MERCADORIA NO ESTOQUE - SUSPEITA FURTO,601,1.10,"R$ 14,18",DIEGO
1,03/01/2026,HORTI,PODRE DESCARTADO NO LIXO,601,6.92,"R$ 100,71",BETAO
2,04/01/2026,AÇOUGUE,LATA AMASSADA NO DESCARREGAMENTO,1000001,7.95,"R$ 98,53",BETAO
3,06/01/2026,CARNES,DIFERENCA DE PESO NA BALANCA,501,1.07,"R$ 19,3",DESCONHECIDO
4,07/01/2026,HORTI,SOBRA DE PAO DORMIDO,601,4.59,"R$ 158,44",DIEGO


---

### Trata origem

In [16]:
silverPerdas['SETOR_ORIGEM'].value_counts()

SETOR_ORIGEM
DEPOSITO           10
AÇOUGUE             7
HORTIFRUTI          7
HORTI               6
CARNES              6
PADARIA             5
FRENTE DE CAIXA     5
Name: count, dtype: int64

In [17]:
# silverPerdas['SETOR_ORIGEM'] = (
#     silverPerdas['SETOR_ORIGEM']
#     .str.replace('Ç', 'C', regex=True)
#     .str.replace('HORTI', 'HORTIFRUTI')
#     .str.replace('CARNES', 'ACOUGUE')
#     .str.replace('frente de caixa', 'CAIXA')
# )


silverPerdas['SETOR_ORIGEM'] = (
      silverPerdas['SETOR_ORIGEM']
      .str.replace('Ç', 'C', regex=False)
      .replace({'HORTI': 'HORTIFRUTI', 'CARNES': 'ACOUGUE', 'FRENTE DE CAIXA': 'CAIXA'})
  )

In [18]:
silverPerdas.head(5)

,DATA_OCORRENCIA,SETOR_ORIGEM,DESCRICAO,COD_PROD,QTD_AVARIADA,PREJUIZO_ESTIMADO,RESPONSAVEL
0,01/01/2026,DEPOSITO,FALTA DE MERCADORIA NO ESTOQUE - SUSPEITA FURTO,601,1.10,"R$ 14,18",DIEGO
1,03/01/2026,HORTIFRUTI,PODRE DESCARTADO NO LIXO,601,6.92,"R$ 100,71",BETAO
2,04/01/2026,ACOUGUE,LATA AMASSADA NO DESCARREGAMENTO,1000001,7.95,"R$ 98,53",BETAO
3,06/01/2026,ACOUGUE,DIFERENCA DE PESO NA BALANCA,501,1.07,"R$ 19,3",DESCONHECIDO
4,07/01/2026,HORTIFRUTI,SOBRA DE PAO DORMIDO,601,4.59,"R$ 158,44",DIEGO


In [19]:
silverPerdas['SETOR_ORIGEM'].value_counts()

SETOR_ORIGEM
HORTIFRUTI    13
ACOUGUE       13
DEPOSITO      10
PADARIA        5
CAIXA          5
Name: count, dtype: int64

-----

### COD_PRODUTO

In [20]:
silverPerdas.value_counts()

DATA_OCORRENCIA  SETOR_ORIGEM  DESCRICAO                                        COD_PROD  QTD_AVARIADA  PREJUIZO_ESTIMADO  RESPONSAVEL 
01/01/2026       DEPOSITO      FALTA DE MERCADORIA NO ESTOQUE - SUSPEITA FURTO  601       1.10          R$ 14,18           DIEGO           1
03/01/2026       HORTIFRUTI    PODRE DESCARTADO NO LIXO                         601       6.92          R$ 100,71          BETAO           1
04/01/2026       ACOUGUE       LATA AMASSADA NO DESCARREGAMENTO                 1000001   7.95          R$ 98,53           BETAO           1
06/01/2026       ACOUGUE       DIFERENCA DE PESO NA BALANCA                     501       1.07          R$ 19,3            DESCONHECIDO    1
07/01/2026       HORTIFRUTI    SOBRA DE PAO DORMIDO                             601       4.59          R$ 158,44          DIEGO           1
10/01/2026       ACOUGUE       QUEBRA DE CAIXA OPERADORA JULIANA                101       1.13          R$ 136,15          JULIANA         1
12/01/2026       P

### chama cadatro

In [21]:
DF_Cadastro = pd.read_csv("..\\Raw\\RAW_CADASTRO_PRODUTOS.csv")

In [22]:
DF_Cadastro.head(10)

,COD_PROD,CATEGORIA_BRUTA,NOME_PRODUTO,VALOR_VENDA_CADASTRO,CUSTO_CADASTRO
0,101,HORTIFRUTI,Tomate Debora kg,"7,90",3.2
1,102,Horti Fruti,BANANA PRATA KG,5.5,"2,10"
2,103,Hortifruti,Alface Americana un,3.5,1.2
3,104,HORTI,Batata Monalisa kg,5.9,2.5
4,105,Hortifruti,Cebola Nacional kg,NaN,2.8
5,201,Carnes,Picanha Bov Resfriada kg,79.9,42.0
6,202,CARNES,Alcatra c/ Maminha kg,44.9,26.0
7,203,Carnes & Açougue,Carne Moida de 2a kg,29.9,14.0
8,204,Carnes,Linguica Toscana kg,22.9,12.0
9,204,Carnes,Linguica Toscana kg,22.9,12.0


In [23]:
# # Busca qualquer produto na DF_Cadastro que contenha "PAO" (ignorando maiúsculas/minúsculas)
# df_pao = DF_Cadastro.loc[DF_Cadastro['NOME_PRODUTO'].astype(str).str.contains('PAO', case=False, na=False)]
# df_pao

In [24]:
# df_picanha = DF_Cadastro.loc[DF_Cadastro['NOME_PRODUTO'].astype(str).str.contains('PICANHA', case=False, na=False)]
# df_picanha

In [25]:
silverPerdas['RESPONSAVEL'].value_counts()

RESPONSAVEL
BETAO           13
DESCONHECIDO    10
DIEGO            9
CARLOS           8
JULIANA          6
Name: count, dtype: int64

In [26]:
silverPerdas['RESPONSAVEL'] = (
    silverPerdas['RESPONSAVEL']
    .str.strip()
    .fillna('DESC')
    .str.replace('DESCONHECIDO', 'DESC') 
    .str.upper()
)

In [27]:
silverPerdas.head(5)

,DATA_OCORRENCIA,SETOR_ORIGEM,DESCRICAO,COD_PROD,QTD_AVARIADA,PREJUIZO_ESTIMADO,RESPONSAVEL
0,01/01/2026,DEPOSITO,FALTA DE MERCADORIA NO ESTOQUE - SUSPEITA FURTO,601,1.10,"R$ 14,18",DIEGO
1,03/01/2026,HORTIFRUTI,PODRE DESCARTADO NO LIXO,601,6.92,"R$ 100,71",BETAO
2,04/01/2026,ACOUGUE,LATA AMASSADA NO DESCARREGAMENTO,1000001,7.95,"R$ 98,53",BETAO
3,06/01/2026,ACOUGUE,DIFERENCA DE PESO NA BALANCA,501,1.07,"R$ 19,3",DESC
4,07/01/2026,HORTIFRUTI,SOBRA DE PAO DORMIDO,601,4.59,"R$ 158,44",DIEGO


-------

### AJUSTA DATA

#### Cria data com datetime

In [28]:
# 1. Converte a coluna original de data
silverPerdas['DATA'] = pd.to_datetime(silverPerdas['DATA_OCORRENCIA'], format='%d/%m/%Y')

# 2. Extrai apenas o ano como número inteiro (2026)
silverPerdas['ANO'] = silverPerdas['DATA'].dt.year
silverPerdas['MES'] = silverPerdas['DATA'].dt.month
silverPerdas['DIA'] = silverPerdas['DATA'].dt.day
silverPerdas.head(5)

,DATA_OCORRENCIA,SETOR_ORIGEM,DESCRICAO,COD_PROD,QTD_AVARIADA,PREJUIZO_ESTIMADO,RESPONSAVEL,DATA,ANO,MES,DIA
0,01/01/2026,DEPOSITO,FALTA DE MERCADORIA NO ESTOQUE - SUSPEITA FURTO,601,1.10,"R$ 14,18",DIEGO,2026-01-01,2026,1,1
1,03/01/2026,HORTIFRUTI,PODRE DESCARTADO NO LIXO,601,6.92,"R$ 100,71",BETAO,2026-01-03,2026,1,3
2,04/01/2026,ACOUGUE,LATA AMASSADA NO DESCARREGAMENTO,1000001,7.95,"R$ 98,53",BETAO,2026-01-04,2026,1,4
3,06/01/2026,ACOUGUE,DIFERENCA DE PESO NA BALANCA,501,1.07,"R$ 19,3",DESC,2026-01-06,2026,1,6
4,07/01/2026,HORTIFRUTI,SOBRA DE PAO DORMIDO,601,4.59,"R$ 158,44",DIEGO,2026-01-07,2026,1,7


In [29]:
silverPerdas.drop('DATA_OCORRENCIA', axis=1).head(5)

,SETOR_ORIGEM,DESCRICAO,COD_PROD,QTD_AVARIADA,PREJUIZO_ESTIMADO,RESPONSAVEL,DATA,ANO,MES,DIA
0,DEPOSITO,FALTA DE MERCADORIA NO ESTOQUE - SUSPEITA FURTO,601,1.10,"R$ 14,18",DIEGO,2026-01-01,2026,1,1
1,HORTIFRUTI,PODRE DESCARTADO NO LIXO,601,6.92,"R$ 100,71",BETAO,2026-01-03,2026,1,3
2,ACOUGUE,LATA AMASSADA NO DESCARREGAMENTO,1000001,7.95,"R$ 98,53",BETAO,2026-01-04,2026,1,4
3,ACOUGUE,DIFERENCA DE PESO NA BALANCA,501,1.07,"R$ 19,3",DESC,2026-01-06,2026,1,6
4,HORTIFRUTI,SOBRA DE PAO DORMIDO,601,4.59,"R$ 158,44",DIEGO,2026-01-07,2026,1,7


In [30]:

silverPerdas['DATA'].info()

<class 'pandas.Series'>
RangeIndex: 46 entries, 0 to 45
Series name: DATA
Non-Null Count  Dtype         
--------------  -----         
46 non-null     datetime64[us]
dtypes: datetime64[us](1)
memory usage: 500.0 bytes


----

### VALORES AJUSTE

In [31]:
# silverPerdas['PREJUIZO_ESTIMADO'] = silverPerdas['PREJUIZO_ESTIMADO'].str.replace("R$ ","")
# silverPerdas['V_PREJUIZO'] = silverPerdas['PREJUIZO_ESTIMADO'].str.replace(",",".")
# silverPerdas['V_PREJUIZO'] = pd.to_numeric(silverPerdas['V_PREJUIZO'], errors='coerce')

silverPerdas['V_PREJUIZO'] = pd.to_numeric(
      silverPerdas['PREJUIZO_ESTIMADO'].astype(str)
          .str.replace('R$', '', regex=False).str.strip()
          .str.replace(',', '.', regex=False),
      errors='coerce'
  ).fillna(0.0)

In [32]:
silverPerdas['V_PREJUIZO'] .info()

<class 'pandas.Series'>
RangeIndex: 46 entries, 0 to 45
Series name: V_PREJUIZO
Non-Null Count  Dtype  
--------------  -----  
46 non-null     float64
dtypes: float64(1)
memory usage: 500.0 bytes


In [33]:
silverPerdas.head(5)

,DATA_OCORRENCIA,SETOR_ORIGEM,DESCRICAO,COD_PROD,QTD_AVARIADA,PREJUIZO_ESTIMADO,RESPONSAVEL,DATA,ANO,MES,DIA,V_PREJUIZO
0,01/01/2026,DEPOSITO,FALTA DE MERCADORIA NO ESTOQUE - SUSPEITA FURTO,601,1.10,"R$ 14,18",DIEGO,2026-01-01,2026,1,1,14.18
1,03/01/2026,HORTIFRUTI,PODRE DESCARTADO NO LIXO,601,6.92,"R$ 100,71",BETAO,2026-01-03,2026,1,3,100.71
2,04/01/2026,ACOUGUE,LATA AMASSADA NO DESCARREGAMENTO,1000001,7.95,"R$ 98,53",BETAO,2026-01-04,2026,1,4,98.53
3,06/01/2026,ACOUGUE,DIFERENCA DE PESO NA BALANCA,501,1.07,"R$ 19,3",DESC,2026-01-06,2026,1,6,19.30
4,07/01/2026,HORTIFRUTI,SOBRA DE PAO DORMIDO,601,4.59,"R$ 158,44",DIEGO,2026-01-07,2026,1,7,158.44


-------

In [34]:
resultado = False

In [35]:
def export_dados_excel(resultado):

    pasta = Path("..\\Raw\\table_silver\\")
    pasta.mkdir(parents=True, exist_ok=True)

    if resultado:

        arquivo_excel = pasta / "silverPerdas_ETL.xlsx"

        # 1. Criar os arquivos
        silverPerdas.to_excel(arquivo_excel, index=False)

        # 3. Consolidar em um Excel com duas abas
        with pd.ExcelWriter(arquivo_excel, engine="openpyxl") as writer:

            silverPerdas.to_excel(
                writer,
                sheet_name="Excel",
                index=False
            )

        return True

    print("ERRO")
    return False

In [36]:
export_dados_excel(True)

True